# Principal Component Analysis -- UCI Dry Bean Dataset

This notebook applies PCA to the UCI Dry Bean dataset (`fetch_ucirepo(id=602)`), which
contains 13,611 dried bean grains photographed under controlled conditions and described by
16 morphological features (area, perimeter, axis lengths, shape factors, etc.). The target
is the bean variety: one of seven cultivars commonly grown in Turkey.

- **Explained variance:** quantify how much information each principal component captures
- **2-D projection:** visualise the 16-dimensional feature space compressed to 2 dimensions
- **Reconstruction error:** show how reconstruction quality changes as components are dropped
- **Component loadings:** inspect which morphological features each principal axis weights most
- All dimensionality reduction uses `mlpackage.PCA`; no sklearn PCA is used

## Mathematical Intuition

### Covariance Matrix

Given a mean-centred data matrix $X \in \mathbb{R}^{n \times d}$, the covariance matrix is:

$$C = \frac{1}{n} X^\top X \in \mathbb{R}^{d \times d}$$

### Eigendecomposition

PCA finds the orthonormal eigenvectors $\mathbf{v}_1, \ldots, \mathbf{v}_d$ of $C$ ordered by
decreasing eigenvalue $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d \geq 0$.
Each eigenvalue equals the variance of the data projected onto the corresponding eigenvector.
In practice the decomposition is computed via SVD: $X = U \Sigma V^\top$, where the columns of
$V$ are the principal axes and the singular values $\sigma_i = \sqrt{n \lambda_i}$.

### Projection and Reconstruction

The first $k$ components form a projection matrix $V_k \in \mathbb{R}^{d \times k}$.
The low-dimensional embedding is $Z = X V_k$ and the reconstruction is $\hat X = Z V_k^\top$.
Reconstruction error decreases monotonically as $k$ grows.

## Dataset Overview

**Source:** UCI Dry Bean (`fetch_ucirepo(id=602)`) | **Rows:** 13,611 | **Features:** 16
morphological attributes from segmented bean images | **Target:** bean variety (7 classes)

| Class | Count |
|---|---|
| DERMASON | 3,546 |
| SIRA     | 2,636 |
| SEKER    | 2,027 |
| HOROZ    | 1,928 |
| CALI     | 1,630 |
| BARBUNYA | 1,322 |
| BOMBAY   |   522 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from ucimlrepo import fetch_ucirepo
from mlpackage import PCA, StandardScaler, train_test_split

bean = fetch_ucirepo(id=602)
X_df = bean.data.features
y_str = bean.data.targets.iloc[:, 0].values

# Map class strings to integer codes (sorted alphabetically for stable colors)
classes_sorted = sorted(set(y_str))
class_to_int = {c: i for i, c in enumerate(classes_sorted)}
y_raw = np.array([class_to_int[c] for c in y_str])
X_raw = X_df.values.astype(float)

print(f"Shape   : {X_raw.shape}")
print(f"Classes : {classes_sorted}")
print(f"Counts  : {dict(zip(*np.unique(y_str, return_counts=True)))}")

## Exploratory Data Analysis

In [ ]:
counts = pd.Series(y_str).value_counts().reindex(classes_sorted)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(classes_sorted, counts.values, color="steelblue")
axes[0].set_title("Bean Class Distribution")
axes[0].set_xlabel("Variety")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)

# Distribution of feature variances (note: features have very different scales)
feature_vars = X_raw.var(axis=0)
axes[1].bar(np.arange(len(feature_vars)), feature_vars, color="coral")
axes[1].set_title("Per-Feature Variance (raw, before scaling)")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("Variance")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  |  Test: {X_test_s.shape}")

## Fitting PCA

In [ ]:
pca_full = PCA(n_components=X_train_s.shape[1])
pca_full.fit(X_train_s)

evr        = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)

print(f"Components to explain 80%: {np.searchsorted(cumulative, 0.80) + 1}")
print(f"Components to explain 90%: {np.searchsorted(cumulative, 0.90) + 1}")
print(f"Components to explain 95%: {np.searchsorted(cumulative, 0.95) + 1}")

## Visualizations

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

n_show = X_train_s.shape[1]
ax1.bar(np.arange(1, n_show + 1), evr, color="steelblue", alpha=0.7, label="Individual")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(np.arange(1, n_show + 1), cumulative, color="coral", linewidth=2, label="Cumulative")
ax2.axhline(0.80, color="green",  linestyle="--", linewidth=1, label="80%")
ax2.axhline(0.90, color="orange", linestyle="--", linewidth=1, label="90%")
ax2.axhline(0.95, color="red",    linestyle="--", linewidth=1, label="95%")
ax2.set_ylabel("Cumulative Variance Explained", color="coral")
ax2.tick_params(axis="y", labelcolor="coral")
ax2.set_ylim(0, 1.05)

plt.title("Explained Variance: Individual (bars) and Cumulative (line)")
fig.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(evr) + 1), evr, marker="o", color="steelblue", linewidth=2)
plt.title("Scree Plot")
plt.xlabel("Component Index")
plt.ylabel("Explained Variance Ratio")
plt.tight_layout()
plt.show()

In [ ]:
pca_2d = PCA(n_components=2)
pca_2d.fit(X_train_s)
Z_train = pca_2d.transform(X_train_s)

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(10, 7))
for cls_i, cls_name in enumerate(classes_sorted):
    mask = y_train == cls_i
    plt.scatter(Z_train[mask, 0], Z_train[mask, 1],
                color=cmap(cls_i), s=10, alpha=0.6, label=cls_name)
plt.title("2-D PCA Projection of Dry Bean Train Set")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend(title="Variety", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruction error on test set as a function of n_components
n_components_list = [1, 2, 3, 5, 8, 10, 12, 16]
errors = []
for k in n_components_list:
    pca_k = PCA(n_components=k)
    pca_k.fit(X_train_s)
    X_rec = pca_k.transform(X_test_s) @ pca_k.components_
    mse   = float(np.mean((X_test_s - X_rec) ** 2))
    errors.append(mse)

plt.figure(figsize=(8, 5))
plt.plot(n_components_list, errors, marker="o", color="steelblue", linewidth=2)
plt.title("Reconstruction MSE vs Number of Components (Test Set)")
plt.xlabel("Number of components")
plt.ylabel("Mean Squared Reconstruction Error")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

for k, e in zip(n_components_list, errors):
    print(f"k={k:2d}   MSE = {e:.6f}")

In [ ]:
# Component loadings -- which features each PC weights heavily
feature_names = list(X_df.columns)
components    = pca_full.components_  # (16, 16)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.ravel()):
    ax.barh(np.arange(len(feature_names)), components[i], color="steelblue")
    ax.set_yticks(np.arange(len(feature_names)))
    ax.set_yticklabels(feature_names, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(f"PC {i+1} (var={evr[i]:.3f})", fontsize=10)
plt.suptitle("First 8 Principal Components -- Feature Loadings")
plt.tight_layout()
plt.show()

## Interpretation and Conclusions

_Analysis to be completed after running the notebook end-to-end._